# 🧪 Insurance Fabric Accelerator — Demo Data Generator

Generates synthetic insurance data across ALL domains for demos and testing.
Creates realistic data including edge cases, fraud scenarios, and regulatory scenarios.

**Fabric Features Used:**
- `spark` (pre-initialized) — DataFrame API
- Delta Lake — write to Lakehouse tables
- `notebookutils.fs` — file operations in OneLake

**No `SparkSession` import needed — Fabric provides `spark` automatically.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Parameters (set via pipeline or notebookutils.notebook.run)        ║
# ╚══════════════════════════════════════════════════════════════════════╝
RECORD_SCALE = 1000       # Multiply all counts by this factor
INCLUDE_FRAUD = True      # Generate fraud scenarios
INCLUDE_EDGE_CASES = True # Generate edge cases
TARGET_LAKEHOUSE = "bronze_demo"

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Imports — only PySpark functions (spark is pre-initialized)        ║
# ╚══════════════════════════════════════════════════════════════════════╝
from pyspark.sql.functions import (
    col, lit, rand, randn, floor, ceil, when, concat, concat_ws,
    date_add, date_sub, current_date, current_timestamp, expr,
    round as spark_round, array, element_at, sha2, substring,
    monotonically_increasing_id, to_date, year, month, lpad,
    from_json, to_json, struct, explode, sequence, row_number
)
from pyspark.sql.types import *
from pyspark.sql.window import Window
import uuid

print(f"✅ Using pre-initialized spark (v{spark.version}) — no SparkSession import")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Reference Data — used to generate realistic values                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

STATES = ['AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN',
          'IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV',
          'NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN',
          'TX','UT','VT','VA','WA','WV','WI','WY']

FIRST_NAMES = ['James','Mary','Robert','Patricia','John','Jennifer','Michael','Linda',
               'David','Elizabeth','William','Barbara','Richard','Susan','Joseph','Jessica',
               'Thomas','Sarah','Charles','Karen','Maria','Jose','Carlos','Ana','Luis']

LAST_NAMES = ['Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis',
              'Rodriguez','Martinez','Hernandez','Lopez','Gonzalez','Wilson','Anderson',
              'Thomas','Taylor','Moore','Jackson','Martin','Lee','Perez','Thompson','White']

PRODUCTS = [
    ('PROD001','Auto - Liability','P&C','auto'),
    ('PROD002','Auto - Comprehensive','P&C','auto'),
    ('PROD003','Homeowners - HO3','P&C','property'),
    ('PROD004','Renters Insurance','P&C','property'),
    ('PROD005','Term Life 20','Life','life'),
    ('PROD006','Whole Life','Life','life'),
    ('PROD007','Group Health PPO','Health','health'),
    ('PROD008','Individual Health HMO','Health','health'),
    ('PROD009','Workers Comp','P&C','workers_comp'),
    ('PROD010','Commercial General Liability','P&C','commercial'),
    ('PROD011','Professional Liability E&O','P&C','professional'),
    ('PROD012','Umbrella Policy','P&C','umbrella'),
]

LOSS_TYPES = ['collision','theft','fire','water_damage','wind','hail','liability',
              'medical','death','disability','property_damage','vandalism','fraud']

CHANNELS = ['web','mobile','phone','agent','mail','email']

print(f"✅ Reference data loaded: {len(STATES)} states, {len(PRODUCTS)} products")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Generate: Customers (MDM)                                          ║
# ╚══════════════════════════════════════════════════════════════════════╝

num_customers = 10 * RECORD_SCALE

# Use Spark-native random generation — no external libraries needed
customers_df = (
    spark.range(0, num_customers)
    .withColumn("customer_id", concat(lit("CUST"), lpad(col("id").cast("string"), 8, "0")))
    .withColumn("first_name", element_at(
        array(*[lit(n) for n in FIRST_NAMES]),
        (floor(rand() * len(FIRST_NAMES)) + 1).cast("int")))
    .withColumn("last_name", element_at(
        array(*[lit(n) for n in LAST_NAMES]),
        (floor(rand() * len(LAST_NAMES)) + 1).cast("int")))
    .withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))
    .withColumn("date_of_birth", date_sub(
        current_date(), (floor(rand() * 18250 + 6570)).cast("int")))  # 18-68 years old
    .withColumn("gender", when(rand() < 0.5, "M").otherwise("F"))
    .withColumn("ssn", concat(
        lpad(floor(rand() * 900 + 100).cast("string"), 3, "0"), lit("-"),
        lpad(floor(rand() * 90 + 10).cast("string"), 2, "0"), lit("-"),
        lpad(floor(rand() * 9000 + 1000).cast("string"), 4, "0")))
    .withColumn("email", concat(
        col("first_name"), lit("."), col("last_name"),
        floor(rand() * 999).cast("string"), lit("@example.com")))
    .withColumn("phone", concat(
        lit("("), lpad(floor(rand() * 900 + 200).cast("string"), 3, "0"),
        lit(") "), lpad(floor(rand() * 900 + 100).cast("string"), 3, "0"),
        lit("-"), lpad(floor(rand() * 9000 + 1000).cast("string"), 4, "0")))
    .withColumn("state_code", element_at(
        array(*[lit(s) for s in STATES]),
        (floor(rand() * len(STATES)) + 1).cast("int")))
    .withColumn("zip_code", lpad(floor(rand() * 90000 + 10000).cast("string"), 5, "0"))
    .withColumn("customer_since_date", date_sub(
        current_date(), (floor(rand() * 3650 + 30)).cast("int")))
    .withColumn("customer_segment",
        when(rand() < 0.1, "platinum")
        .when(rand() < 0.3, "gold")
        .when(rand() < 0.6, "silver")
        .otherwise("standard"))
    .withColumn("preferred_channel", element_at(
        array(*[lit(c) for c in CHANNELS]),
        (floor(rand() * len(CHANNELS)) + 1).cast("int")))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .drop("id")
)

customers_df.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.customer_raw")
print(f"✅ Generated {num_customers:,} customers")
customers_df.show(5, truncate=False)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Generate: Agents & Brokers                                         ║
# ╚══════════════════════════════════════════════════════════════════════╝

num_agents = max(RECORD_SCALE // 10, 50)

agents_df = (
    spark.range(0, num_agents)
    .withColumn("agent_id", concat(lit("AGT"), lpad(col("id").cast("string"), 5, "0")))
    .withColumn("agent_name", concat(
        element_at(array(*[lit(n) for n in FIRST_NAMES]),
                   (floor(rand() * len(FIRST_NAMES)) + 1).cast("int")),
        lit(" "),
        element_at(array(*[lit(n) for n in LAST_NAMES]),
                   (floor(rand() * len(LAST_NAMES)) + 1).cast("int"))))
    .withColumn("agent_type", when(rand() < 0.6, "captive").otherwise("independent"))
    .withColumn("license_state", element_at(
        array(*[lit(s) for s in STATES]),
        (floor(rand() * len(STATES)) + 1).cast("int")))
    .withColumn("commission_rate", spark_round(rand() * 0.12 + 0.03, 3))
    .withColumn("tenure_years", floor(rand() * 25 + 1).cast("int"))
    .withColumn("is_active", when(rand() < 0.9, True).otherwise(False))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .drop("id")
)

agents_df.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.agent_raw")
print(f"✅ Generated {num_agents} agents")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Generate: Policies (Life / Health / P&C)                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

num_policies = 20 * RECORD_SCALE

# Create products reference
products_df = spark.createDataFrame(PRODUCTS,
    ["product_code", "product_name", "lob", "sub_lob"])
products_df.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.product_catalog_raw")

policies_df = (
    spark.range(0, num_policies)
    .withColumn("policy_id", concat(lit("POL"), lpad(col("id").cast("string"), 9, "0")))
    .withColumn("policy_number", concat(
        lit("INS-"), lpad(floor(rand() * 9000000 + 1000000).cast("string"), 7, "0")))
    .withColumn("customer_id", concat(
        lit("CUST"), lpad(floor(rand() * num_customers).cast("string"), 8, "0")))
    .withColumn("agent_id", concat(
        lit("AGT"), lpad(floor(rand() * num_agents).cast("string"), 5, "0")))
    .withColumn("_prod_idx", (floor(rand() * len(PRODUCTS)) + 1).cast("int"))
    .withColumn("product_code", element_at(
        array(*[lit(p[0]) for p in PRODUCTS]), col("_prod_idx")))
    .withColumn("product_name", element_at(
        array(*[lit(p[1]) for p in PRODUCTS]), col("_prod_idx")))
    .withColumn("line_of_business", element_at(
        array(*[lit(p[2]) for p in PRODUCTS]), col("_prod_idx")))
    .withColumn("effective_date", date_sub(
        current_date(), (floor(rand() * 730)).cast("int")))
    .withColumn("expiration_date", date_add(col("effective_date"), 365))
    .withColumn("issue_date", date_sub(col("effective_date"), (floor(rand() * 14 + 1)).cast("int")))
    .withColumn("policy_status",
        when(rand() < 0.70, "active")
        .when(rand() < 0.85, "expired")
        .when(rand() < 0.92, "cancelled")
        .when(rand() < 0.97, "pending")
        .otherwise("lapsed"))
    .withColumn("premium_amount", spark_round(
        when(col("line_of_business") == "P&C", rand() * 4000 + 500)
        .when(col("line_of_business") == "Life", rand() * 2000 + 200)
        .otherwise(rand() * 8000 + 1000), 2))
    .withColumn("coverage_amount", spark_round(
        when(col("line_of_business") == "Life", rand() * 900000 + 100000)
        .otherwise(rand() * 450000 + 50000), 0))
    .withColumn("deductible_amount",
        element_at(array(lit(250.0), lit(500.0), lit(1000.0), lit(2500.0), lit(5000.0)),
                   (floor(rand() * 5) + 1).cast("int")))
    .withColumn("state_code", element_at(
        array(*[lit(s) for s in STATES]),
        (floor(rand() * len(STATES)) + 1).cast("int")))
    .withColumn("risk_class",
        when(rand() < 0.3, "preferred")
        .when(rand() < 0.7, "standard")
        .otherwise("substandard"))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .drop("id", "_prod_idx")
)

policies_df.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.policy_raw")
print(f"✅ Generated {num_policies:,} policies")
policies_df.groupBy("line_of_business", "policy_status").count().orderBy("line_of_business").show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Generate: Claims (Normal + Fraud Scenarios)                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

num_claims = 5 * RECORD_SCALE
fraud_rate = 0.08  # 8% fraud rate

claims_df = (
    spark.range(0, num_claims)
    .withColumn("claim_id", concat(lit("CLM"), lpad(col("id").cast("string"), 9, "0")))
    .withColumn("claim_number", concat(
        lit("CL-"), lpad(floor(rand() * 9000000 + 1000000).cast("string"), 7, "0")))
    .withColumn("policy_id", concat(
        lit("POL"), lpad(floor(rand() * num_policies).cast("string"), 9, "0")))
    .withColumn("customer_id", concat(
        lit("CUST"), lpad(floor(rand() * num_customers).cast("string"), 8, "0")))
    .withColumn("loss_date", date_sub(
        current_date(), (floor(rand() * 365 + 1)).cast("int")))
    .withColumn("report_date", date_add(col("loss_date"),
        (floor(rand() * 30)).cast("int")))
    .withColumn("loss_type", element_at(
        array(*[lit(l) for l in LOSS_TYPES]),
        (floor(rand() * len(LOSS_TYPES)) + 1).cast("int")))
    .withColumn("cause_of_loss", concat(lit("Cause: "), col("loss_type")))
    .withColumn("loss_description", concat(
        lit("Incident involving "), col("loss_type"), lit(" at insured location.")))
    .withColumn("loss_amount_estimate", spark_round(
        when(col("loss_type") == "death", rand() * 500000 + 50000)
        .when(col("loss_type") == "fire", rand() * 200000 + 10000)
        .when(col("loss_type") == "liability", rand() * 100000 + 5000)
        .otherwise(rand() * 30000 + 500), 2))
    .withColumn("loss_location_state", element_at(
        array(*[lit(s) for s in STATES]),
        (floor(rand() * len(STATES)) + 1).cast("int")))
    .withColumn("claim_status",
        when(rand() < 0.35, "closed")
        .when(rand() < 0.60, "open")
        .when(rand() < 0.75, "under_investigation")
        .when(rand() < 0.85, "pending_documents")
        .when(rand() < 0.95, "approved")
        .otherwise("denied"))
    .withColumn("channel", element_at(
        array(*[lit(c) for c in ['web','mobile','phone','agent']]),
        (floor(rand() * 4) + 1).cast("int")))
    # Fraud indicators
    .withColumn("is_fraud", when(rand() < fraud_rate, True).otherwise(False))
    .withColumn("fraud_score",
        when(col("is_fraud"), spark_round(rand() * 0.3 + 0.7, 3))  # High score for fraud
        .otherwise(spark_round(rand() * 0.4, 3)))                    # Low score for legit
    .withColumn("total_paid",
        when(col("claim_status").isin("closed", "approved"),
             spark_round(col("loss_amount_estimate") * (rand() * 0.3 + 0.6), 2))
        .otherwise(lit(0.0)))
    .withColumn("total_reserved",
        when(col("claim_status").isin("open", "under_investigation"),
             spark_round(col("loss_amount_estimate") * 1.1, 2))
        .otherwise(lit(0.0)))
    .withColumn("litigation_flag", when(rand() < 0.05, True).otherwise(False))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .drop("id")
)

claims_df.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.claim_raw")

fraud_count = claims_df.filter(col("is_fraud")).count()
print(f"✅ Generated {num_claims:,} claims ({fraud_count} fraudulent = {fraud_count/num_claims:.1%})")
claims_df.groupBy("claim_status").count().orderBy("count", ascending=False).show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Generate: Premium Invoices & Payments                              ║
# ╚══════════════════════════════════════════════════════════════════════╝

num_invoices = 15 * RECORD_SCALE

invoices_df = (
    spark.range(0, num_invoices)
    .withColumn("invoice_id", concat(lit("INV"), lpad(col("id").cast("string"), 9, "0")))
    .withColumn("policy_id", concat(
        lit("POL"), lpad(floor(rand() * num_policies).cast("string"), 9, "0")))
    .withColumn("customer_id", concat(
        lit("CUST"), lpad(floor(rand() * num_customers).cast("string"), 8, "0")))
    .withColumn("billing_period_start", date_sub(
        current_date(), (floor(rand() * 365)).cast("int")))
    .withColumn("billing_period_end", date_add(col("billing_period_start"), 30))
    .withColumn("due_date", date_add(col("billing_period_start"), 15))
    .withColumn("invoice_amount", spark_round(rand() * 2000 + 100, 2))
    .withColumn("payment_status",
        when(rand() < 0.65, "paid")
        .when(rand() < 0.80, "pending")
        .when(rand() < 0.90, "overdue")
        .when(rand() < 0.95, "grace")
        .otherwise("lapsed"))
    .withColumn("paid_amount",
        when(col("payment_status") == "paid", col("invoice_amount"))
        .when(col("payment_status") == "grace",
              spark_round(col("invoice_amount") * (rand() * 0.5 + 0.3), 2))
        .otherwise(lit(0.0)))
    .withColumn("payment_method",
        element_at(array(lit("card"), lit("ach"), lit("check"), lit("wire")),
                   (floor(rand() * 4) + 1).cast("int")))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .drop("id")
)

invoices_df.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.premium_invoice_raw")
print(f"✅ Generated {num_invoices:,} invoices")
invoices_df.groupBy("payment_status").count().orderBy("count", ascending=False).show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Generate: Finance / Journal Entries                                ║
# ╚══════════════════════════════════════════════════════════════════════╝

num_entries = 30 * RECORD_SCALE
ACCOUNTS = [
    ('4010','Premium Revenue','revenue'),('4020','Earned Premium','revenue'),
    ('5010','Claims Expense','expense'),('5020','Commission Expense','expense'),
    ('5030','Operating Expense','expense'),('1010','Cash','asset'),
    ('1020','Investments','asset'),('1030','Premiums Receivable','asset'),
    ('2010','Loss Reserves','liability'),('2020','Unearned Premium','liability'),
]

journal_df = (
    spark.range(0, num_entries)
    .withColumn("entry_id", concat(lit("JE"), lpad(col("id").cast("string"), 10, "0")))
    .withColumn("posting_date", date_sub(current_date(), (floor(rand() * 365)).cast("int")))
    .withColumn("fiscal_year", year(col("posting_date")))
    .withColumn("fiscal_period", month(col("posting_date")))
    .withColumn("_acct_idx", (floor(rand() * len(ACCOUNTS)) + 1).cast("int"))
    .withColumn("account_code", element_at(
        array(*[lit(a[0]) for a in ACCOUNTS]), col("_acct_idx")))
    .withColumn("account_name", element_at(
        array(*[lit(a[1]) for a in ACCOUNTS]), col("_acct_idx")))
    .withColumn("account_category", element_at(
        array(*[lit(a[2]) for a in ACCOUNTS]), col("_acct_idx")))
    .withColumn("amount", spark_round(rand() * 50000 + 100, 2))
    .withColumn("debit_amount",
        when(col("account_category").isin("asset", "expense"), col("amount")).otherwise(lit(0.0)))
    .withColumn("credit_amount",
        when(col("account_category").isin("liability", "revenue"), col("amount")).otherwise(lit(0.0)))
    .withColumn("source_module",
        element_at(array(lit("premium"), lit("claims"), lit("commission"), lit("reinsurance")),
                   (floor(rand() * 4) + 1).cast("int")))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .drop("id", "_acct_idx", "amount")
)

journal_df.write.format("delta").mode("overwrite").saveAsTable(f"{TARGET_LAKEHOUSE}.journal_entry_raw")
print(f"✅ Generated {num_entries:,} journal entries")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Summary: All Generated Data                                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n" + "="*60)
print("  📊 DEMO DATA GENERATION SUMMARY")
print("="*60)

tables = spark.sql(f"SHOW TABLES IN {TARGET_LAKEHOUSE}").collect()
total_rows = 0
for t in tables:
    tbl = f"{TARGET_LAKEHOUSE}.{t['tableName']}"
    cnt = spark.table(tbl).count()
    total_rows += cnt
    print(f"  {t['tableName']:30s} {cnt:>12,} rows")

print(f"  {'─'*42}")
print(f"  {'TOTAL':30s} {total_rows:>12,} rows")
print("="*60)

# Exit value for orchestration
notebookutils.notebook.exit(f"Generated {total_rows} rows across {len(tables)} tables")